# Lab Assignment 4

MST Dependency Parsing and Training

**Total: 20 marks**

## Objective
Implement core components of an MST dependency parser, analyse a Chu-Liu/Edmonds cycle, and perform one structured-perceptron update using gold and predicted dependency trees.

## Submission
- Complete all `TODO` sections and all markdown answers.
- Do not modify the supplied data or expected-test cells.
- Run **Kernel > Restart & Run All** before submitting the completed `.ipynb` file.

## Valid dependency tree conditions
Each non-root word has exactly one incoming head; `ROOT` has none; all words are reachable from `ROOT`; and no directed cycle occurs.

In [34]:
from itertools import product
from collections import defaultdict, Counter
import pandas as pd
import matplotlib.pyplot as plt

## Task 1: Validate Dependency Trees (2 marks)

Implement `is_valid_tree()` to verify whether a set of dependency arcs forms a valid rooted dependency tree.

A valid dependency tree must satisfy all of the following conditions:

1. `ROOT` has no incoming arc.
2. Every non-root word has exactly one incoming head.
3. No word can be its own head (no self-loop).
4. The graph must not contain a directed cycle.
5. Every word must be reachable from `ROOT`.
6. The number of arcs must be exactly `number_of_tokens - 1`.

### Your task

Complete the `is_valid_tree()` function below.

Do not change the test cases.

Your function should return:

- `True` for a valid dependency tree.
- `False` for an invalid dependency structure.

You should ensure that the function correctly detects duplicate heads, missing heads, cycles, self-loops, disconnected components, and arcs entering `ROOT`.

In [35]:
def is_valid_tree(tokens, arcs, root="ROOT"):
    # TODO: Implement all rooted dependency-tree checks.
    non_root = [w for w in tokens if w != root]

    if len(arcs) != len(non_root):
        return False

    heads = {}
    for head, dep in arcs:
        if head == dep or dep == root:
            return False
        if dep in heads:
            return False
        heads[dep] = head

    if set(heads) != set(non_root):
        return False

    for start in non_root:
        cur = start
        seen = set()
        while cur != root:
            if cur in seen or cur not in heads:
                return False
            seen.add(cur)
            cur = heads[cur]

    return True

test_tokens = ["ROOT", "A", "B", "C"]
valid = [("ROOT","A"), ("A","B"), ("A","C")]
cycle = [("ROOT","A"), ("B","C"), ("C","B")]
duplicate = [("ROOT","A"), ("A","B"), ("C","B")]
print(is_valid_tree(test_tokens, valid))      # Expected: True
print(is_valid_tree(test_tokens, cycle))      # Expected: False
print(is_valid_tree(test_tokens, duplicate))  # Expected: False

True
False
False


## Task 2: Exhaustive MST Reference Parser (3 marks)

Implement an exhaustive reference parser for a small dependency graph.

For every non-root word:

1. Enumerate all possible incoming heads.
2. Construct candidate dependency trees.
3. Use `is_valid_tree()` from Task 1 to reject invalid structures.
4. Compute the total score of every valid tree.
5. Return the highest-scoring tree and its score.

The score of a tree is:

$$
Score(G)=\sum_{(h,d)\in G} Score(h\rightarrow d)
$$

where:

- $h$ = head
- $d$ = dependent
- $G$ = dependency tree

### Your task

Complete:

```python
exhaustive_mst(tokens, score)

In [36]:

tokens = ["ROOT", "Researchers", "analyze", "data", "carefully"]

score = {
    ("ROOT","Researchers"):1,
    ("ROOT","analyze"):7,
    ("ROOT","data"):1,
    ("ROOT","carefully"):1,

    ("Researchers","analyze"):2,
    ("Researchers","data"):2,
    ("Researchers","carefully"):2,

    ("analyze","Researchers"):6,
    ("analyze","data"):7,
    ("analyze","carefully"):6,

    ("data","Researchers"):2,
    ("data","analyze"):3,
    ("data","carefully"):3,

    ("carefully","Researchers"):2,
    ("carefully","analyze"):3,
    ("carefully","data"):4,
}

def exhaustive_mst(tokens, score, root="ROOT"):
    # TODO:
    # 1. Generate candidate heads for every non-root node.
    # 2. Construct candidate trees.
    # 3. Keep only valid trees using is_valid_tree().
    # 4. Calculate total tree score.
    # 5. Return the highest-scoring tree.

    non_root = [w for w in tokens if w != root]
    choices = [[h for h in tokens if h != dep] for dep in non_root]
    best_score = float("-inf")
    best_tree = None

    for heads in product(*choices):
        tree = list(zip(heads, non_root))
        if is_valid_tree(tokens, tree, root):
            total = sum(score[a] for a in tree)
            if total > best_score:
                best_score = total
                best_tree = tree

    return best_score, best_tree

best_score, best_arcs = exhaustive_mst(tokens, score)

print("Best score:", best_score)
print("Best arcs:", best_arcs)

Best score: 26
Best arcs: [('analyze', 'Researchers'), ('ROOT', 'analyze'), ('analyze', 'data'), ('analyze', 'carefully')]


## Task 3: Chu-Liu/Edmonds — Best Incoming Edges and Cycle Detection (3 marks)

The Chu-Liu/Edmonds algorithm starts by selecting the highest-scoring incoming edge for every non-root node.

For each node $v$:

$$
parent(v)
=
\arg\max_{u\neq v} Score(u\rightarrow v)
$$

If the selected edges form a valid tree, the algorithm is finished.

However, if the selected edges contain a directed cycle, the cycle must be detected and contracted before continuing.

### Your task

Complete the following two functions:

1. `best_incoming()`
2. `find_cycle()`

`best_incoming()` should select the highest-scoring incoming edge for every non-root node.

`find_cycle()` should:

- return one directed cycle as a list of nodes if a cycle exists;
- return `None` if no cycle exists.

### Questions

After running the code:

1. What edges were selected by the greedy incoming-edge step?
2. Which nodes form the cycle?
3. Why is the greedy result not a valid dependency tree?
4. Why can we not simply accept the greedy result as the MST?

In [37]:
cycle_tokens = ["ROOT", "A", "B", "C"]

cycle_score = {
    ("ROOT","A"):5,
    ("ROOT","B"):4,
    ("ROOT","C"):3,

    ("A","B"):10,
    ("B","A"):10,

    ("A","C"):7,
    ("B","C"):8,

    ("C","A"):2,
    ("C","B"):2,
}


def best_incoming(tokens, score, root="ROOT"):
    # TODO:
    # For every non-root node:
    #   find the incoming edge with maximum score
    # Do not allow edges entering ROOT.
    greedy = {}
    for dep in tokens:
        if dep == root:
            continue
        candidates = [
            (head, dep, score[(head, dep)])
            for head in tokens
            if head != dep
        ]
        greedy[dep] = max(candidates, key=lambda x: x[2])
    """ print("Greedy incoming edges:")
    for edge in greedy.values():
        print(edge) """
    return greedy    


def find_cycle(arcs):
    # TODO:
    # Detect a directed cycle in the selected arcs.
    # Return the cycle as a list of nodes.
    # Return None if no cycle exists.
    
    # Build parent map from arcs dictionary
    # arcs is {dep: (head, dep, score)}
    parent = {}
    for dep, (head, _, _) in arcs.items():
        parent[dep] = head
    
    # For each node, follow parent pointers to detect cycle
    for start in parent:
        visited = set()
        current = start
        path = []
        
        while current is not None:
            if current in visited:
                # Found a cycle - extract it from path
                cycle_start_idx = path.index(current)
                return path[cycle_start_idx:]
            
            visited.add(current)
            path.append(current)
            current = parent.get(current)
    
    return None


greedy_arcs = best_incoming(cycle_tokens, cycle_score)

print("Greedy arcs:", greedy_arcs)
print("Cycle:", find_cycle(greedy_arcs))

Greedy arcs: {'A': ('B', 'A', 10), 'B': ('A', 'B', 10), 'C': ('B', 'C', 8)}
Cycle: ['A', 'B']


**Task 3 explanation:**

**1. What edges were selected by the greedy incoming-edge step?**

The greedy algorithm selects the maximum-scoring incoming edge for each non-root node:
- **A**: Receives edge from B with score 10 → (B, A, 10)
- **B**: Receives edge from A with score 10 → (A, B, 10)
- **C**: Receives edge from B with score 8 → (B, C, 8)

**2. Which nodes form the cycle?**

Nodes **A** and **B** form a directed cycle: **A → B → A** (a cycle of length 2).

**3. Why is the greedy result not a valid dependency tree?**

The greedy selected edges contain a directed cycle (A ↔ B), which violates the acyclicity requirement for dependency trees. A valid rooted tree must be acyclic. Additionally, although C has an incoming edge, the presence of the cycle prevents proper tree structure. The graph fails the acyclicity condition which is essential for any spanning arborescence.

**4. Why can we not simply accept the greedy result as the MST?**

The greedy algorithm selects the best incoming edge independently for each node. This works for undirected graphs or general directed graphs, but for rooted dependency trees (arborescences), we need to ensure both acyclicity AND connectivity to the root. When a cycle exists, the greedy solution is not a valid tree. Simply removing edges from cycles arbitrarily would not guarantee optimality. The Chu-Liu/Edmonds algorithm solves this by detecting cycles, contracting them into a single node with adjusted edge weights, solving recursively, and then expanding the solution back to the original graph while maintaining optimality.

## Task 4: Chu-Liu/Edmonds Cycle Contraction (3 marks)

Suppose the greedy step produces the cycle:

$$
A\rightarrow B
$$

and

$$
B\rightarrow A
$$

with:

$$
Score(A\rightarrow B)=10
$$

$$
Score(B\rightarrow A)=10
$$

The total score of the selected cycle is:

$$
CycleScore=10+10=20
$$

When an external edge enters a node inside the cycle, the cycle is contracted into a single **supernode**.

For an external edge $h\rightarrow d$ entering the cycle, calculate the adjusted score as:

$$
\boxed{
AdjustedScore(h\rightarrow d)
=
Score(h\rightarrow d)
+
CycleScore
-
Score(old\_incoming\_edge(d))
}
$$

### Example

For:

$$
ROOT\rightarrow A
$$

we have:

$$
Score(ROOT\rightarrow A)=5
$$

The selected edge entering `A` inside the cycle is:

$$
B\rightarrow A
$$

with score:

$$
Score(B\rightarrow A)=10
$$

Therefore:

$$
AdjustedScore(ROOT\rightarrow A)
=
5+20-10
=
15
$$

### Your task

Complete `entering_cycle_table()`.

Create a table for:

- `ROOT -> A`
- `ROOT -> B`

The table must contain:

- candidate edge
- original score
- selected cycle edge being removed
- removed edge score
- total cycle score
- adjusted score

### Interpretation question

Which candidate edge should be selected to enter the contracted cycle?

Explain which original cycle edge will be removed when the contracted tree is expanded back to the original graph.

In [38]:
def entering_cycle_table(score, cycle, selected_cycle_arcs):
    # TODO:
    # Create a DataFrame containing:
    #   outside_head
    #   cycle_dependent
    #   original_score
    #   removed_cycle_arc
    #   removed_score
    #   cycle_total
    #   adjusted_score
    
    # Calculate total cycle score
    cycle_total = sum(score[arc] for arc in selected_cycle_arcs)
    
    # Build a map of cycle members to their incoming cycle edges
    cycle_incoming = {}
    for head, dep in selected_cycle_arcs:
        cycle_incoming[dep] = (head, dep)
    
    rows = []
    # Find all external edges entering the cycle
    for node in ["ROOT"]:
        for cycle_node in cycle:
            if (node, cycle_node) in score:
                original = score[(node, cycle_node)]
                removed_arc = cycle_incoming[cycle_node]
                removed = score[removed_arc]
                adjusted = original + cycle_total - removed
                
                rows.append({
                    "outside_head": node,
                    "cycle_dependent": cycle_node,
                    "original_score": original,
                    "removed_cycle_arc": removed_arc,
                    "removed_score": removed,
                    "cycle_total": cycle_total,
                    "adjusted_score": adjusted
                })
    
    return pd.DataFrame(rows)


selected_cycle_arcs = [
    ("B","A"),
    ("A","B")
]

entering_cycle_table(
    cycle_score,
    ["A", "B"],
    selected_cycle_arcs
)

,outside_head,cycle_dependent,original_score,removed_cycle_arc,removed_score,cycle_total,adjusted_score
0,ROOT,A,5,"(B, A)",10,20,15
1,ROOT,B,4,"(A, B)",10,20,14


## Task 5: Implement the Chu-Liu/Edmonds MST Parser (4 marks)

In this task, you will implement the **Chu-Liu/Edmonds algorithm from scratch** for finding the maximum-scoring rooted directed spanning tree.

Do **not** use:

- `networkx.maximum_spanning_arborescence`
- any other library implementation of Chu-Liu/Edmonds

You may use Python data structures such as dictionaries, lists, and sets.

### Algorithm

Your implementation should follow these steps:

#### Step 1: Select the best incoming edge

For every non-root node, select the incoming edge with the highest score.

#### Step 2: Check for cycles

If the selected edges form a valid rooted tree, return them.

If a directed cycle exists, continue.

#### Step 3: Identify the cycle

Find one directed cycle:

$$
C=\{v_1,v_2,\ldots,v_k\}
$$

#### Step 4: Contract the cycle

Replace all nodes in the cycle by a single supernode.

For an edge entering the cycle:

$$
u\rightarrow v,\qquad v\in C
$$

use:

$$
w'(u\rightarrow C)
=
w(u\rightarrow v)
+
W_C
-
w(parent(v)\rightarrow v)
$$

where $W_C$ is the total score of the selected cycle edges.

For an edge leaving the cycle:

$$
v\rightarrow u,\qquad v\in C
$$

keep the maximum-scoring corresponding outgoing edge.

#### Step 5: Recursively solve the contracted graph

Run the same procedure on the contracted graph.

#### Step 6: Expand the cycle

Use the selected incoming edge to determine which cycle edge must be removed.

Restore the remaining cycle edges and return the final maximum spanning arborescence.

### Required functions

Complete:

```python
chu_liu_edmonds()

In [39]:

def select_best_incoming(nodes, edges, root="ROOT"):
    """
    Select the highest-scoring incoming edge for every non-root node.

    edges:
        dictionary {(head, dependent): score}
    """
    # TODO
    selected = {}

    for dep in nodes:
        if dep == root:
            continue

        candidates = [
            (head, dep)
            for head in nodes
            if head != dep and (head, dep) in edges
        ]

        if not candidates:
            raise ValueError(f"No incoming edge for {dep}")

        best = max(candidates, key=lambda arc: edges[arc])
        selected[dep] = best

    return selected



def detect_cycle(arcs, root="ROOT"):
    """
    Return one directed cycle from arcs.
    Return None if no cycle exists.
    """
    # TODO
    parent = {dep: head for head, dep in arcs}

    for start in parent:
        if start == root:
            continue

        current = start
        path = []
        position = {}

        while current in parent and current != root:
            if current in position:
                return path[position[current]:]

            position[current] = len(path)
            path.append(current)
            current = parent[current]

    return None


def contract_cycle(nodes, edges, selected_arcs, cycle, root="ROOT"):
    """
    Contract the detected cycle into a single supernode.

    Return:
        contracted_nodes
        contracted_edges
        mapping_information

    mapping_information is needed later to expand
    the contracted solution back to the original graph.
    """
    # TODO
    cycle_set = set(cycle)

    # Use a tuple so the supernode is hashable and unique.
    supernode = ("CYCLE", tuple(sorted(cycle_set, key=str)))

    contracted_nodes = [
        node for node in nodes
        if node not in cycle_set
    ]
    contracted_nodes.append(supernode)

    # Score of the selected edges inside the cycle
    cycle_total = sum(edges[arc] for arc in selected_arcs
                      if arc[0] in cycle_set and arc[1] in cycle_set)

    # Original selected incoming edge for each cycle node
    old_incoming = {
        dep: arc
        for arc in selected_arcs
        if arc[1] in cycle_set
        for dep in [arc[1]]
    }

    contracted_edges = {}
    mapping_information = {
        "supernode": supernode,
        "cycle": cycle_set,
        "incoming": {},
        "outgoing": {}
    }

    for (head, dep), weight in edges.items():

        head_in = head in cycle_set
        dep_in = dep in cycle_set

        # Ignore edges entirely inside the cycle
        if head_in and dep_in:
            continue

        # Edge entering the cycle
        if not head_in and dep_in:
            new_arc = (head, supernode)

            old_arc = old_incoming[dep]
            adjusted = weight + cycle_total - edges[old_arc]

            # Keep the best edge if several enter the cycle
            if (
                new_arc not in contracted_edges
                or adjusted > contracted_edges[new_arc]
            ):
                contracted_edges[new_arc] = adjusted

                mapping_information["incoming"][new_arc] = {
                    "original_arc": (head, dep),
                    "removed_cycle_arc": old_arc
                }

        # Edge leaving the cycle
        elif head_in and not dep_in:
            new_arc = (supernode, dep)

            # Keep the best outgoing edge
            if (
                new_arc not in contracted_edges
                or weight > contracted_edges[new_arc]
            ):
                contracted_edges[new_arc] = weight

                mapping_information["outgoing"][new_arc] = {
                    "original_arc": (head, dep)
                }

        else:
            new_arc = (head, dep)

            if (
                new_arc not in contracted_edges
                or weight > contracted_edges[new_arc]
            ):
                contracted_edges[new_arc] = weight

    return contracted_nodes, contracted_edges, mapping_information



def expand_cycle(
    selected_contracted_arcs,
    cycle,
    mapping_information,
    original_selected_arcs
):
    """
    Expand the contracted solution back to the original graph.

    The incoming edge selected for the supernode determines
    which cycle edge must be removed.
    """
    # TODO
    supernode = mapping_information["supernode"]

    # Start with all the original cycle edges.
    cycle_set = set(cycle)

    result = [
        arc for arc in original_selected_arcs
        if arc[0] in cycle_set and arc[1] in cycle_set
    ]

    # Process the selected edges from the contracted solution.
    for arc in selected_contracted_arcs:

        # Edge entering the cycle
        if arc[1] == supernode:
            info = mapping_information["incoming"][arc]

            # Add the actual external edge
            result.append(info["original_arc"])

            # Remove the cycle edge entering that node
            removed = info["removed_cycle_arc"]
            result = [a for a in result if a != removed]

        # Edge leaving the cycle
        elif arc[0] == supernode:
            info = mapping_information["outgoing"][arc]
            result.append(info["original_arc"])

        # Edge completely outside the cycle
        else:
            result.append(arc)

    return result


def chu_liu_edmonds(tokens, score, root="ROOT"):
    """
    Implement the Chu-Liu/Edmonds maximum spanning
    arborescence algorithm from scratch.

    Return:
        total_score, final_arcs
    """

    def solve(nodes, edges, root):
        # TODO 1:
        # Select the highest-scoring incoming edge
        # for every non-root node.
        selected_dict = select_best_incoming(nodes, edges, root)
        selected_arcs = list(selected_dict.values())
        # TODO 2:
        # Detect whether the selected edges contain a cycle.
        cycle = detect_cycle(selected_arcs, root)

        # TODO 3:
        # If there is no cycle, return the selected tree.
        if cycle is None:
            return selected_arcs
    
        # TODO 4:
        # If there is a cycle, contract it.
        contracted_nodes, contracted_edges, mapping = contract_cycle(
            nodes,
            edges,
            selected_arcs,
            cycle,
            root
        )

    
        # TODO 5:
        # Recursively solve the contracted graph.
        contracted_solution = solve(
            contracted_nodes,
            contracted_edges,
            root
        )
    
        # TODO 6:
        # Expand the cycle.
        expanded = expand_cycle(
            contracted_solution,
            cycle,
            mapping,
            selected_arcs
        )

        return expanded

    final_arcs = solve(
        list(tokens),
        score,
        root
    )

    total_score = sum(score[arc] for arc in final_arcs)

    return total_score, final_arcs
    
# Test on the supplied graph
cle_score, cle_arcs = chu_liu_edmonds(tokens, score)

print("Chu-Liu/Edmonds score:", cle_score)
print("Chu-Liu/Edmonds arcs:", cle_arcs)

print(
    "Matches exhaustive MST score?",
    cle_score == best_score
)

Chu-Liu/Edmonds score: 26
Chu-Liu/Edmonds arcs: [('analyze', 'Researchers'), ('ROOT', 'analyze'), ('analyze', 'data'), ('analyze', 'carefully')]
Matches exhaustive MST score? True


## Task 6: Inference-Based Learning with Structured Perceptron (5 marks)

In this task, you will train the MST dependency parser using **inference-based structured perceptron learning**.

The parser uses the current weight vector to score every possible dependency tree and selects the highest-scoring tree:

$$
\boxed{
G_{pred}
=
\arg\max_{G\in T(x)}
\mathbf{w}\cdot\Phi(x,G)
}
$$

where:

- $x$ = input sentence
- $T(x)$ = set of valid dependency trees for the sentence
- $\mathbf{w}$ = current feature-weight vector
- $\Phi(x,G)$ = feature vector of dependency tree $G$

The predicted tree is then compared with the gold tree.

If:

$$
G_{pred}\neq G_{gold}
$$

update the weights using:

$$
\boxed{
\mathbf{w}_{new}
=
\mathbf{w}_{old}
+
\Phi(x,G_{gold})
-
\Phi(x,G_{pred})
}
$$

Equivalently, for each feature:

$$
\boxed{
w_{new}[f]
=
w_{old}[f]
+
\Phi(x,G_{gold})[f]
-
\Phi(x,G_{pred})[f]
}
$$

### Training procedure

Your training loop must perform the following steps:

1. Initialize the feature weights.
2. Take one training sentence and its gold dependency tree.
3. Compute the score of candidate dependency arcs using the current weights.
4. Run your **own Chu-Liu/Edmonds implementation from Task 5** to obtain the predicted tree.
5. Compute the feature vector of the gold tree.
6. Compute the feature vector of the predicted tree.
7. Update the weights using the structured perceptron rule.
8. Repeat for all training examples.
9. Repeat the complete process for multiple epochs.

### Important

The training procedure must call your implementation from **Task 5**.

Do not use:

```python
networkx.maximum_spanning_arborescence()

In [40]:

from collections import Counter

# ---------------------------------------------------------
# Training data
# ---------------------------------------------------------

training_data = [

    {
        "tokens": ["ROOT", "John", "saw", "Mary"],
        "pos": {
            "ROOT": "ROOT",
            "John": "NOUN",
            "saw": "VERB",
            "Mary": "NOUN"
        },
        "gold": [
            ("ROOT", "saw"),
            ("saw", "John"),
            ("saw", "Mary")
        ]
    },

    {
        "tokens": ["ROOT", "Dogs", "chase", "cats"],
        "pos": {
            "ROOT": "ROOT",
            "Dogs": "NOUN",
            "chase": "VERB",
            "cats": "NOUN"
        },
        "gold": [
            ("ROOT", "chase"),
            ("chase", "Dogs"),
            ("chase", "cats")
        ]
    },

    {
        "tokens": ["ROOT", "Alice", "likes", "Bob"],
        "pos": {
            "ROOT": "ROOT",
            "Alice": "NOUN",
            "likes": "VERB",
            "Bob": "NOUN"
        },
        "gold": [
            ("ROOT", "likes"),
            ("likes", "Alice"),
            ("likes", "Bob")
        ]
    }
]


# ---------------------------------------------------------
# Feature extraction
# ---------------------------------------------------------

def arc_features(tokens, pos, head, dep):
    """
    Return a list of feature names for one dependency arc.

    Required feature templates:
      1. head word
      2. dependent word
      3. head POS
      4. dependent POS
      5. direction
      6. distance
      7. head-dependent word pair
    """

    # TODO
    # Do not simply return the score.
    # Return feature names/counts for this arc.

    h_index = tokens.index(head)
    d_index = tokens.index(dep)

    direction = "RIGHT" if h_index < d_index else "LEFT"
    distance = abs(h_index - d_index)

    return [
        ("HEAD_WORD", head),
        ("DEP_WORD", dep),
        ("HEAD_POS", pos[head]),
        ("DEP_POS", pos[dep]),
        ("DIRECTION", direction),
        ("DISTANCE", distance),
        ("WORD_PAIR", head, dep)
    ]


def tree_feature_vector(tokens, pos, tree):
    """
    Sum the feature vectors of all arcs in a dependency tree.

    Return a Counter or dictionary:
        feature -> count
    """

    # TODO
    features = Counter()

    for head, dep in tree:
        features.update(
            arc_features(tokens, pos, head, dep)
        )

    return features


# ---------------------------------------------------------
# Arc and tree scoring
# ---------------------------------------------------------

def score_arc(tokens, pos, head, dep, weights):
    """
    Score one dependency arc:

        score(h -> d) = w . f(h -> d)

    """

    # TODO
    features = arc_features(tokens, pos, head, dep)

    return sum(
        weights[feature]
        for feature in features
    )


def build_score_graph(tokens, pos, weights):
    """
    Construct the complete directed graph of candidate
    dependency arcs.

    Return:
        {(head, dependent): score}
    """

    # TODO
    score_graph = {}

    for dep in tokens:
        if dep == "ROOT":
            continue

        for head in tokens:
            if head == dep:
                continue

            score_graph[(head, dep)] = score_arc(
                tokens,
                pos,
                head,
                dep,
                weights
            )

    return score_graph



def score_tree(tokens, pos, tree, weights):
    """
    Return the total score of a dependency tree.
    """

    # TODO
    return sum(
        score_arc(tokens, pos, head, dep, weights)
        for head, dep in tree
    )

# ---------------------------------------------------------
# Structured perceptron update
# ---------------------------------------------------------

def perceptron_update(
    weights,
    gold_features,
    predicted_features,
    learning_rate=1.0
):
    """
    Structured perceptron update:

        w_new =
        w_old
        + eta * Phi(gold)
        - eta * Phi(predicted)
    """

    # TODO
    new_weights = weights.copy()

    # Add gold features
    for feature, count in gold_features.items():
        new_weights[feature] += learning_rate * count

    # Subtract predicted features
    for feature, count in predicted_features.items():
        new_weights[feature] -= learning_rate * count

    return new_weights


# ---------------------------------------------------------
# Inference-based training
# ---------------------------------------------------------

def train_perceptron(
    training_data,
    num_epochs=5,
    learning_rate=1.0
):
    """
    Train the MST parser using inference-based
    structured perceptron learning.

    For every epoch:

        1. Infer predicted tree using Task 5.
        2. Compare with gold tree.
        3. Update weights if prediction is incorrect.
        4. Record number of mistakes.
    """

    # Initialize all weights to zero.
    weights = Counter()

    history = []

    for epoch in range(num_epochs):

        mistakes = 0

        for example in training_data:

            tokens = example["tokens"]
            pos = example["pos"]
            gold_tree = example["gold"]

            # ---------------------------------------------
            # TODO 1:
            # Build arc scores using current weights.
            # ---------------------------------------------

            score_graph = build_score_graph(
                tokens,
                pos,
                weights
            )

            # ---------------------------------------------
            # TODO 2:
            # Run YOUR Task 5 implementation.
            # ---------------------------------------------

            predicted_score, predicted_tree = chu_liu_edmonds(
                tokens,
                score_graph
            )

            # ---------------------------------------------
            # TODO 3:
            # Compare predicted and gold trees.
            # ---------------------------------------------

            if set(predicted_tree) != set(gold_tree):

                mistakes += 1

                # -----------------------------------------
                # TODO 4:
                # Compute gold feature vector.
                # -----------------------------------------

                gold_features = tree_feature_vector(
                    tokens,
                    pos,
                    gold_tree
                )

                # -----------------------------------------
                # TODO 5:
                # Compute predicted feature vector.
                # -----------------------------------------

                predicted_features = tree_feature_vector(
                    tokens,
                    pos,
                    predicted_tree
                )

                # -----------------------------------------
                # TODO 6:
                # Update weights.
                # -----------------------------------------

                weights = perceptron_update(
                    weights,
                    gold_features,
                    predicted_features,
                    learning_rate
                )

        history.append(mistakes)

        print(
            f"Epoch {epoch + 1}: "
            f"{mistakes} mistake(s)"
        )

    return weights, history

In [41]:
# Run the complete inference-based training

final_weights, training_history = train_perceptron(
    training_data,
    num_epochs=5,
    learning_rate=1.0
)

print("\nTraining history:")
for epoch, mistakes in enumerate(training_history, start=1):
    print(
        f"Epoch {epoch}: "
        f"{mistakes} mistake(s)"
    )

print("\nLearned weights:")
for feature, weight in sorted(final_weights.items()):
    print(f"{feature}: {weight}")

Epoch 1: 1 mistake(s)
Epoch 2: 0 mistake(s)
Epoch 3: 0 mistake(s)
Epoch 4: 0 mistake(s)
Epoch 5: 0 mistake(s)

Training history:
Epoch 1: 1 mistake(s)
Epoch 2: 0 mistake(s)
Epoch 3: 0 mistake(s)
Epoch 4: 0 mistake(s)
Epoch 5: 0 mistake(s)

Learned weights:
('DEP_POS', 'NOUN'): 0.0
('DEP_POS', 'VERB'): 0.0
('DEP_WORD', 'John'): 0.0
('DEP_WORD', 'Mary'): 0.0
('DEP_WORD', 'saw'): 0.0
('DIRECTION', 'LEFT'): 1.0
('DIRECTION', 'RIGHT'): -1.0
('DISTANCE', 1): 1.0
('DISTANCE', 2): 0.0
('DISTANCE', 3): -1.0
('HEAD_POS', 'ROOT'): -2.0
('HEAD_POS', 'VERB'): 2.0
('HEAD_WORD', 'ROOT'): -2.0
('HEAD_WORD', 'saw'): 2.0
('WORD_PAIR', 'ROOT', 'John'): -1.0
('WORD_PAIR', 'ROOT', 'Mary'): -1.0
('WORD_PAIR', 'ROOT', 'saw'): 0.0
('WORD_PAIR', 'saw', 'John'): 1.0
('WORD_PAIR', 'saw', 'Mary'): 1.0


## Task 6: Training Analysis

### Q1. Prediction Errors

For each epoch, report the number of incorrectly predicted trees.

| Epoch | Number of Mistakes |
|------:|-------------------:|
| 1 | |
| 2 | |
| 3 | |
| 4 | |
| 5 | |

### Q2. Weight Updates

Identify at least two features whose weights increased during training.

Explain why these features received a positive update.

### Q3. Negative Updates

Identify at least two features whose weights decreased.

Explain why these features received a negative update.

### Q4. Effect of the Perceptron Update

Explain how:

$$
\mathbf{w}_{new}
=
\mathbf{w}_{old}
+
\Phi(x,G_{gold})
-
\Phi(x,G_{pred})
$$

makes the gold dependency tree more likely to be selected during the next inference step.

### Q5. Training Behaviour

Did the number of mistakes decrease across epochs?

If it did, explain why.

If it did not, give a possible reason based on the feature representation or training data.

## Task 6 (ANSWERS)

### A1. Prediction Errors

For each epoch, report the number of incorrectly predicted trees.

| Epoch | Number of Mistakes |
|------:|-------------------:|
| 1 | 1|
| 2 | 0|
| 3 | 0|
| 4 | 0|
| 5 | 0|

### A2. Weight Updates

- Two features whose weights increased are:

    * ('DIRECTION', 'LEFT'), which increased to 1.0. This feature received a positive update because it occurred more often in the gold tree than in the incorrectly predicted tree during the update.
    * ('HEAD_POS', 'VERB'), which increased to 2.0. This indicates that the gold trees contained more dependencies headed by verbs than the predicted tree during the mistake, so the perceptron increased its weight.
 

### A3. Negative Updates

- Two features whose weights decreased are:

    * ('DIRECTION', 'RIGHT'), which decreased to -1.0. It received a negative update because it occurred more in the predicted tree than in the gold tree during the mistake.
    * ('HEAD_POS', 'ROOT'), which decreased to -2.0. This means the predicted tree had more arcs with ROOT as the head for the relevant feature counts than the gold tree, so the perceptron penalized this feature.

### A4. Effect of the Perceptron Update

- The update:
$$
\mathbf{w}_{new}
=
\mathbf{w}_{old}
+
\Phi(x,G_{gold})
-
\Phi(x,G_{pred})
$$

increases the weights of features that occur more strongly in the gold tree and decreases the weights of features associated with the predicted tree.

As a result, when the parser performs inference again, the gold tree receives a higher score relative to the incorrect tree. Repeating this process allows the parser to learn which features are useful for selecting the correct dependency structure.
  

### A5. Training Behaviour

- Yes. The number of mistakes decreased from 1 in the first epoch to 0 in the second epoch, and remained at 0 for the remaining epochs.

This happened because the perceptron updated the weights after the incorrect prediction in the first epoch. The updated weights favored features associated with the gold dependency trees and penalized features associated with the incorrect prediction. Consequently, the parser was able to select the correct trees in later epochs.